In [1]:
import sys
sys.path.append('../../Simulate/')

from UtilityFunctions import retrieve_iupac

In [2]:
import subprocess
import numpy as np

from typing import Dict

In [3]:
class StreamHTSIM:
    '''
    stream HTSIM output for bisulfite reads generation
    :param str  sim_cmd : HTSIM commands for simulation
    :param bool pair_end: pair_end or not
    :rtype None
    '''
    def __init__(self, sim_cmd: list = None, pair_end: bool = True):
        self.sim_cmd  = sim_cmd
        self.pair_end = pair_end

    def __iter__(self):
        htsim = subprocess.Popen(self.sim_cmd, stdout=subprocess.PIPE, universal_newlines=True)
        sim_iter = iter(htsim.stdout.readline, b'')

        line  = self.get_line(sim_iter) # line is None when EOF
        while line:
            # collect all variant lines on the contig, after that sim_iter points to read lines
            if line == "Contig Variant Start":
                variant_contig, variant_dict = self.collect_variants(sim_iter)
                yield variant_contig, variant_dict

            # collect read pairs
            for collect_flag, read_pair in self.collect_reads(sim_iter):
                if collect_flag: # {1: collect_reads, 0: swith to collect_vars or EOF}
                    yield False, read_pair
                else:
                    line = "Contig Variant Start" if isinstance(read_pair, list) else None
                    break


    def collect_variants(self, sim_iter):
        '''collect variant lines from stdout'''
        variant_dict = {}
        variant_info = {}

        while True:
            line = self.get_line(sim_iter)
            if line == 'Contig Variant End':
                return variant_info['chrom'], variant_dict

            variant_info = self.process_variant_line(line)
            if variant_info['pos']:
                assert variant_info['pos'] not in variant_dict
                variant_dict[variant_info['pos']] = variant_info


    def collect_reads(self, sim_iter):
        '''collect read lines from stdout'''
        skip_flag = not self.pair_end

        while True:
            line  = self.get_line(sim_iter)
            if not line: # EOF
                yield 0, None
            elif line == "Contig Variant Start": # switch to collect variants
                yield 0, []
            else:
                read1 = self.process_read_lines(sim_iter, line = line)
                read2 = self.process_read_lines(sim_iter, skip = skip_flag)
                yield 1, [read1, read2]


    @staticmethod
    def get_line(sim_iter):
        '''receive lines from console'''
        try:
            line = next(sim_iter).strip()
        except StopIteration:
            print("End of output\n")
            return None
        else:
            return line


    @staticmethod
    def process_variant_line(line: str) -> Dict:
        '''parse variant lines'''
        line_split = line.split('\t')

        try:
            chrom, pos, ref, alt, heter_flag = line_split
        except ValueError:
            return dict(chrom=line_split[0], pos = None)
        else:
            heter = heter_flag == '+'
            indel = int(ref == '-') - int(alt == '-') # 1 for ref=='-', -1 for alt=='-', o.w. 0
            offset= indel * max(len(ref), len(alt))
            if indel:
                iupac  = None
            else:
                iupac  = retrieve_iupac(alt)
                alt    = list(set(iupac) - set(ref))[0]
            return dict(chrom=chrom, pos=int(pos), ref=ref, alt=alt,
                        offset=offset, heter=heter, indel=indel, iupac=iupac)


    @staticmethod
    def process_read_lines(sim_iter, line = None, skip = False):
        '''parse read lines'''
        if skip:
            next(sim_iter)
            next(sim_iter)
            next(sim_iter)
            next(sim_iter)
            return None

        if not line:
            line = next(sim_iter).strip()
        # header, seq, comment process
        read_id, pair, flag_pos, flag_mut, flag_indel, qual, cgr = line.split(' ')
        cgr = np.frombuffer(cgr.encode(), dtype=np.int8)
        seq = np.frombuffer(next(sim_iter).strip().encode(), dtype=np.int8)
        _, start, end, cover_pos, n_sub, n_indel, insert_size, inner_dist, ofs= next(sim_iter).strip().split(':')
        ofs = np.fromstring(ofs, dtype=np.int8, sep = ',')
        ctx = np.frombuffer(next(sim_iter).strip().encode(), np.int8)
        return dict(read_id=read_id, pair=int(pair), qual = int(qual),
                    flag_pos=int(flag_pos), flag_mut=int(flag_mut), flag_indel=int(flag_indel),
                    start=int(start), end=int(end), cover_pos=int(cover_pos),
                    n_sub=int(n_sub), n_indel=int(n_indel),
                    insert_size=int(insert_size), inner_dist=int(inner_dist),
                    cgr=cgr, seq=seq, ofs=ofs, ctx=ctx)


In [4]:
ref_fasta = '/home/wbguo/iproject/BSReadSim/test/data/BSB_test.fa'
ref_fasta_allA  = '/home/wbguo/iproject/BSReadSim/test/data/all_A.fa'
ref_fasta_chr21 = '/home/wbguo/iproject/BSReadSim/test/data/chr21.fa'
ref_fasta_empty = '/home/wbguo/iproject/BSReadSim/test/data/empty.fa'
ref_fasta_nonexist = '/home/wbguo/iproject/BSReadSim/test/data/nonexist.fa'

# Test for different situation

In [5]:
htsim_arg= ['/home/wbguo/iproject/BSReadSim/HTSIM/htsim']

sim_dict = {'-N':1000, 
            '-h':0, 
            '-T':0,
            '-1':100, 
            '-2':100,
            '-e':0.005,
            '-i':400,
            '-I':25,
            '-r':0.001, 
            '-R':0.15,
            '-X':0.15, 
            '-A':0.05,
            '-f':1,
           }

In [6]:
sim_sub = htsim_arg + [str(item) for key_val in sim_dict.items() for item in key_val]

sim_cmd =  sim_sub + [ref_fasta]
sim_cmd_allA = sim_sub + [ref_fasta_allA] 
sim_cmd_chr21= sim_sub + [ref_fasta_chr21] 
sim_cmd_empty_fasta = sim_sub + [ref_fasta_empty]
sim_cmd_nonexist_fasta = sim_sub + [ref_fasta_nonexist]

sim_dict['-r'] = 0
sim_cmd_no_snp  = htsim_arg + [str(item) for key_val in sim_dict.items() for item in key_val] + [ref_fasta]
sim_dict['-r'] = 0.001

sim_dict['-N'] = 10
sim_cmd_small_N = htsim_arg + [str(item) for key_val in sim_dict.items() for item in key_val] + [ref_fasta]
sim_dict['-N'] = 1000

In [7]:
' '.join(sim_cmd_no_snp)

'/home/wbguo/iproject/BSReadSim/HTSIM/htsim -N 1000 -h 0 -T 0 -1 100 -2 100 -e 0.005 -i 400 -I 25 -r 0 -R 0.15 -X 0.15 -A 0.05 -f 1 /home/wbguo/iproject/BSReadSim/test/data/BSB_test.fa'

In [8]:
i = 0
for variant_contig, sim_data in StreamHTSIM(sim_cmd_no_snp):
    if variant_contig:
        print(sim_data)
    
    if variant_contig:
        print(variant_contig)
    if not variant_contig:
        [sim_data[0]['read_id'], sim_data[0]['pair']]
        ++i

Simulating whole genome reads:
Reference genome file: /home/wbguo/iproject/BSReadSim/test/data/BSB_test.fa
[main] Calculating the total length and effective length of the reference sequences...
[main] Found 6 contig sequences, total length: 1961600, effective length: 1961600
[main] No contig id specified, will generate 1000 reads from all contigs
[main] No VCF input, will generate SNP randomly if mutation rate is nonzero
[htsim] seed = 1675918044
[sim_core] contig 'chr10': simulate 215 reads...


{}
chr10


[sim_core] contig 'chr11': simulate 216 reads...


{}
chr11


[sim_core] contig 'chr12': simulate 214 reads...


...

[sim_core] contig 'chr14': simulate 179 reads...


{}
chr14


[sim_core] contig 'chr15': simulate 2 reads...


{}
chr15


[sim_core] Generated 997 read pairs, with 0 contain SNP, 0 contain INDEL


In [9]:
i

0

In [10]:
sim_data[0]

{'read_id': '@chr15:4879:4879:1',
 'pair': 0,
 'qual': 56,
 'flag_pos': 0,
 'flag_mut': 0,
 'flag_indel': 0,
 'start': 4878,
 'end': 4978,
 'cover_pos': 0,
 'n_sub': 0,
 'n_indel': 0,
 'insert_size': 0,
 'inner_dist': -200,
 'cgr': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int8),
 'seq': array([3, 0, 2, 0, 0, 0, 3, 2, 2, 2, 0, 2, 0, 3, 3, 1, 3, 3, 3, 2, 2, 3,
        2, 0, 0, 0, 1, 1, 1, 1, 2, 3, 1, 3, 1, 3, 0, 1, 3, 0, 0, 0, 0, 0,
        3, 0, 1, 0, 0, 0, 0, 0, 0, 3, 3, 0, 2, 1, 1, 2, 2, 2, 1, 2, 3, 2,
        2, 3, 2, 2, 1, 2, 0, 3, 0, 0, 0, 0, 2, 0, 1, 1, 1, 3, 2, 2, 0, 2,
        0, 2, 0, 3, 1, 1, 1, 3, 1, 0, 1, 1], dtype=int8),
 'ofs': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [11]:
import sys
print(sys.getsizeof(sim_data[0]))

640


In [12]:
from pympler import asizeof
print(asizeof.asizeof(sim_data[0]))

2808


# Test the running time and size of core steps

In [13]:
' '.join(sim_cmd)

'/home/wbguo/iproject/BSReadSim/HTSIM/htsim -N 1000 -h 0 -T 0 -1 100 -2 100 -e 0.005 -i 400 -I 25 -r 0.001 -R 0.15 -X 0.15 -A 0.05 -f 1 /home/wbguo/iproject/BSReadSim/test/data/BSB_test.fa'

In [14]:
htsim = subprocess.Popen(sim_cmd, stdout=subprocess.PIPE, universal_newlines=True)
sim_iter = iter(htsim.stdout.readline, b'')

Simulating whole genome reads:
Reference genome file: /home/wbguo/iproject/BSReadSim/test/data/BSB_test.fa
[main] Calculating the total length and effective length of the reference sequences...
[main] Found 6 contig sequences, total length: 1961600, effective length: 1961600
[main] No contig id specified, will generate 1000 reads from all contigs
[main] No VCF input, will generate SNP randomly if mutation rate is nonzero
[htsim] seed = 1675918063
[sim_core] contig 'chr10': simulate 215 reads...


In [15]:
def get_line(sim_iter):
    try:
        line = next(sim_iter).strip()
    except StopIteration:
        print("End of output\n")
        return None
    else:
        return line

In [16]:
def process_variant_line(line: str) -> Dict:
    line_split = line.split('\t')

    try:
        chrom, pos, ref, alt, heter_flag = line_split
    except ValueError:
        return dict(chrom=line_split[0], pos = None)
    else:
        heter = heter_flag == '+'
        indel = int(ref == '-') - int(alt == '-') # 1 for ref=='-', -1 for alt=='-', o.w. 0
        offset= indel * max(len(ref), len(alt))
        if indel:
            iupac  = None
        else:
            iupac  = retrieve_iupac(alt)
            alt    = list(set(iupac) - set(ref))[0]
        return dict(chrom=chrom, pos=int(pos), ref=ref, alt=alt,
                    offset=offset, heter=heter, indel=indel, iupac=iupac)


def collect_variants(sim_iter):
    variant_dict = {}

    while True:
        line = get_line(sim_iter)
        if line == 'Contig Variant End':
            return variant_info['chrom'], variant_dict

        variant_info = process_variant_line(line)
        if variant_info['pos']:
            assert variant_info['pos'] not in variant_dict
            variant_dict[variant_info['pos']] = variant_info

In [17]:
v = collect_variants(sim_iter)

In [18]:
v

('chr10',
 {3815: {'chrom': 'chr10',
   'pos': 3815,
   'ref': '-',
   'alt': 'G',
   'offset': 1,
   'heter': False,
   'indel': 1,
   'iupac': None},
  6596: {'chrom': 'chr10',
   'pos': 6596,
   'ref': '-',
   'alt': 'A',
   'offset': 1,
   'heter': True,
   'indel': 1,
   'iupac': None},
  6711: {'chrom': 'chr10',
   'pos': 6711,
   'ref': 'A',
   'alt': 'C',
   'offset': 0,
   'heter': False,
   'indel': 0,
   'iupac': ('C',)},
  7506: {'chrom': 'chr10',
   'pos': 7506,
   'ref': 'T',
   'alt': 'A',
   'offset': 0,
   'heter': True,
   'indel': 0,
   'iupac': ('A', 'T')},
  8479: {'chrom': 'chr10',
   'pos': 8479,
   'ref': 'T',
   'alt': 'A',
   'offset': 0,
   'heter': True,
   'indel': 0,
   'iupac': ('A', 'T')},
  9876: {'chrom': 'chr10',
   'pos': 9876,
   'ref': 'A',
   'alt': 'G',
   'offset': 0,
   'heter': True,
   'indel': 0,
   'iupac': ('A', 'G')},
  10670: {'chrom': 'chr10',
   'pos': 10670,
   'ref': 'A',
   'alt': 'G',
   'offset': 0,
   'heter': False,
   'indel': 

In [19]:
from pympler import asizeof
print(asizeof.asizeof(v))

220736


In [20]:
len(v[1])

433

In [21]:
while True:
    x = get_line(sim_iter)
    if x[0] == "@":
        break

y = get_line(sim_iter)
z = get_line(sim_iter)
t = get_line(sim_iter)

In [22]:
x

'@chr10:129930:129930:0 0 0 0 0 56 \x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'

In [23]:
x.split(' ')

['@chr10:129930:129930:0',
 '0',
 '0',
 '0',
 '0',
 '56',
 '\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00']

In [24]:
y

'\x02\x02\x01\x01\x00\x00\x00\x01\x03\x00\x00\x01\x03\x03\x00\x01\x00\x03\x03\x01\x01\x00\x00\x00\x03\x02\x00\x00\x03\x02\x00\x02\x03\x00\x02\x02\x02\x00\x03\x03\x00\x00\x02\x03\x02\x02\x00\x01\x00\x03\x03\x03\x03\x03\x01\x01\x03\x03\x03\x00\x03\x03\x02\x02\x00\x03\x03\x03\x03\x03\x01\x00\x01\x00\x03\x02\x02\x00\x00\x03\x00\x01\x03\x00\x01\x00\x01\x00\x02\x03\x01\x00\x03\x00\x03\x00\x00\x00\x00\x02'

In [25]:
z

'+:129929:130029:0:0:0:0:-200:0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0'

In [26]:
t

'\x0f\x0f\x07\x07\x00\x00\x00\x07\x00\x00\x00\x07\x00\x00\x00\x07\x00\x00\x00\x07\x07\x00\x00\x00\x00\x0f\x00\x00\x00\x0f\x00\x0f\x00\x00\x0f\x0f\x0f\x00\x00\x00\x00\x00\x0f\x00\x0f\x0f\x00\x07\x00\x00\x00\x00\x00\x00\x07\x07\x00\x00\x00\x00\x00\x00\x0f\x0f\x00\x00\x00\x00\x00\x00\x07\x00\x07\x00\x00\x0f\x0f\x00\x00\x00\x00\x07\x00\x00\x07\x00\x03\x00\x0b\x00\x07\x00\x00\x00\x00\x00\x00\x00\x00\x0f'

In [27]:
np.frombuffer(t.encode('utf-8'), np.int8)

array([15, 15,  7,  7,  0,  0,  0,  7,  0,  0,  0,  7,  0,  0,  0,  7,  0,
        0,  0,  7,  7,  0,  0,  0,  0, 15,  0,  0,  0, 15,  0, 15,  0,  0,
       15, 15, 15,  0,  0,  0,  0,  0, 15,  0, 15, 15,  0,  7,  0,  0,  0,
        0,  0,  0,  7,  7,  0,  0,  0,  0,  0,  0, 15, 15,  0,  0,  0,  0,
        0,  0,  7,  0,  7,  0,  0, 15, 15,  0,  0,  0,  0,  7,  0,  0,  7,
        0,  3,  0, 11,  0,  7,  0,  0,  0,  0,  0,  0,  0,  0, 15],
      dtype=int8)

In [28]:
#### 12 us & 2968 byte for a read, used the fputc for output
def process_read_name(line_list: list):
    '''parse read lines'''
    # header, seq, comment process
    read_id, pair, flag_pos, flag_mut, flag_indel, qual, cgr = line_list[0].split(' ')
    cgr = np.frombuffer(cgr.encode(), dtype=np.int8)
    seq = np.frombuffer(line_list[1].encode(), dtype=np.int8)
    _, start, end, cover_pos, n_sub, n_indel, insert_size, inner_dist, ofs= line_list[2].split(':')
    ofs = np.fromstring(ofs, dtype=np.int8, sep = ',')
    ctx = np.frombuffer(line_list[3].encode(), np.int8)
    return dict(read_id=read_id, pair=int(pair), qual = int(qual),
                flag_pos=int(flag_pos), flag_mut=int(flag_mut), flag_indel=int(flag_indel),
                start=int(start), end=int(end), cover_pos=int(cover_pos),
                n_sub=int(n_sub), n_indel=int(n_indel),
                insert_size=int(insert_size), inner_dist=int(inner_dist),
                cgr=cgr, seq=seq, ofs=ofs, ctx=ctx)

In [29]:
%timeit process_read_name([x,y,z,t]) 

11.8 µs ± 27.3 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [30]:
import sys
obj = process_read_name([x,y,z,t])
sys.getsizeof(obj)

640

In [31]:
asizeof.asizeof(obj)

2808

In [32]:
obj

{'read_id': '@chr10:129930:129930:0',
 'pair': 0,
 'qual': 56,
 'flag_pos': 0,
 'flag_mut': 0,
 'flag_indel': 0,
 'start': 129929,
 'end': 130029,
 'cover_pos': 0,
 'n_sub': 0,
 'n_indel': 0,
 'insert_size': 0,
 'inner_dist': -200,
 'cgr': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int8),
 'seq': array([2, 2, 1, 1, 0, 0, 0, 1, 3, 0, 0, 1, 3, 3, 0, 1, 0, 3, 3, 1, 1, 0,
        0, 0, 3, 2, 0, 0, 3, 2, 0, 2, 3, 0, 2, 2, 2, 0, 3, 3, 0, 0, 2, 3,
        2, 2, 0, 1, 0, 3, 3, 3, 3, 3, 1, 1, 3, 3, 3, 0, 3, 3, 2, 2, 0, 3,
        3, 3, 3, 3, 1, 0, 1, 0, 3, 2, 2, 0, 0, 3, 0, 1, 3, 0, 1, 0, 1, 0,
        2, 3, 1, 0, 3, 0, 3, 0, 0, 0, 0, 2], dtype=int8),
 'ofs': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [33]:
chr(obj['qual'])

'8'

In [34]:
''.join(['ACGT'[i] for i in obj['seq']])

'GGCCAAACTAACTTACATTCCAAATGAATGAGTAGGGATTAAGTGGACATTTTTCCTTTATTGGATTTTTCACATGGAATACTACACAGTCATATAAAAG'

In [35]:
while True:
    x1 = get_line(sim_iter)
    if x1[0] == "@":
        break

y1 = get_line(sim_iter)
z1 = get_line(sim_iter)
t1 = get_line(sim_iter)

In [36]:
obj1 = process_read_name([x1,y1,z1,t1])

In [37]:
obj1

{'read_id': '@chr10:129930:129930:0',
 'pair': 1,
 'qual': 56,
 'flag_pos': 0,
 'flag_mut': 0,
 'flag_indel': 0,
 'start': 129829,
 'end': 129929,
 'cover_pos': 1,
 'n_sub': 1,
 'n_indel': 0,
 'insert_size': 0,
 'inner_dist': -200,
 'cgr': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int8),
 'seq': array([3, 0, 3, 2, 3, 2, 0, 3, 0, 2, 0, 3, 3, 0, 1, 0, 3, 3, 3, 0, 1, 0,
        0, 2, 3, 0, 2, 1, 1, 1, 0, 3, 3, 1, 0, 0, 0, 0, 1, 3, 2, 2, 0, 3,
        3, 1, 0, 1, 2, 2, 3, 0, 0, 1, 1, 0, 0, 1, 3, 2, 3, 2, 3, 1, 0, 3,
        2, 3, 0, 0, 0, 3, 2, 0, 2, 0, 0, 0, 2, 3, 3, 1, 1, 0, 0, 0, 1, 3,
        2, 1, 3, 3, 3, 1, 1, 0, 1, 0, 2, 2], dtype=int8),
 'ofs': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [38]:
sim_data = [obj, obj1]

In [39]:
sys.getsizeof(sim_data)

72

In [40]:
print(asizeof.asizeof(sim_data))

4648


In [41]:
%timeit obj1['start'] + np.arange(len(obj1['seq']))

3.49 µs ± 203 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [42]:
x = obj1['start'] + np.arange(len(obj1['seq']))

In [43]:
%timeit x + obj1['ofs']

1.47 µs ± 6.86 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)


# Speed & memory test result

In [ ]:
#### 25 us & 640 byte for a read, used the %d for output
ascii_idx = np.array([i for i in range(48,58)] + [i for i in range(97, 103)])
ascii_val = np.array([i for i in range(0,16)])
ascii_arr = np.full(127, -1).astype(np.int8)
ascii_arr[ascii_idx] = ascii_val

def process_read_name2(line_list: list, read_len: int):
    read_id, pair, flag_pos, flag_mut, flag_indel, cgr = line_list[0].split(' ')
    cgr = np.bitwise_and(np.frombuffer(cgr.encode(), dtype=np.int8), 0x03)
    seq = np.bitwise_and(np.frombuffer(line_list[1].encode(), dtype=np.int8), 0x03)
    _, start, end, cover_pos, n_sub, n_indel, insrt_len, insrt_len2, ofs= line_list[2].split(':')
    ofs = np.fromstring(ofs, dtype=np.int8, sep = ',')
    ctx = ascii_arr[np.frombuffer(line_list[3].encode(), np.int8)]
    return dict(read_id=read_id, pair=int(pair), flag_pos=int(flag_pos), flag_mut=int(flag_mut), flag_indel=int(flag_indel),
                start=int(start), end=int(end), cover_pos=int(cover_pos), n_sub=int(n_sub), n_indel=int(n_indel), 
                insrt_len=int(insrt_len), insrt_len2=int(insrt_len2),
                cgr=cgr, seq=seq, ofs=ofs, ctx=ctx)

In [ ]:
#### 38 us & 360 byte for a read, used the %d for output
def process_read_name3(line_list: list, read_len: int):
    arr = np.full([4, read_len], np.NaN)
    read_id, pair, num_var, num_indel, start, end, mut = line_list[0].split(' ')
    arr[0] = np.bitwise_and(np.frombuffer(mut.encode(), dtype=np.int8), 0x03)
    arr[1] = np.bitwise_and(np.frombuffer(line_list[1].encode(), dtype=np.int8), 0x03)
    _, n_sub, n_indel, insrt_len, ofs = line_list[2].split(':')
    arr[2] = np.fromstring(ofs, dtype=np.int8, sep = ',')
    arr[3] = ascii_arr[np.frombuffer(line_list[3].encode(), np.int8)]
    return dict(read_id = read_id, pair = int(pair),  start=int(start), end=int(end), 
                num_var = int(num_var), num_indel = int(num_indel), n_sub = int(n_sub), n_indel = int(n_indel),
                arr = arr)

In [ ]:
### if use %d
%timeit np.fromstring(y, dtype=np.int8)                               # 1.56 us, will give 48-51
%timeit np.frombuffer(y.encode(), dtype=np.int8)                      # 0.9  us, will give 48-51
%timeit np.array(list(y), dtype=np.int8)                              # 13.5 us, will give 0-3
%timeit np.frombuffer(y.encode(), dtype=np.int8) - 48                 # 4.25 us, will give 0-3
%timeit np.bitwise_and(np.frombuffer(y.encode(), dtype=np.int8), 0x3) # 4.38 us, will give 0-3
%timeit ascii_arr[np.frombuffer(y.encode(), dtype=np.int8)]           # 4.8  us, will give 0-3